In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install ultralytics onnx onnxruntime numpy pillow opencv-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 91.5 MB/s eta 0:00:00


In [4]:
from pathlib import Path

best_pt = Path("/content/drive/MyDrive/best.pt")
print("best.pt exists:", best_pt.exists())
print("File size:", best_pt.stat().st_size / 1e6, "MB")

best.pt exists: True
File size: 20.289925 MB


In [5]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/best.pt")

model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    simplify=True,
)

print("Export done")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.3 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 217ms
Prepared 2 packages in 37ms
Installed 2 packages in 7ms
 + colorama==0.4

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 6.1s, saved as '/content/drive/MyDrive/best.onnx' (36.4 MB)

Export complete (7.4s)
Results saved to /content/drive/MyDrive/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/best.onnx imgsz=640 data=/content/data.yaml  
Visualize:       https://netron.app
Export done


In [6]:
import shutil
from pathlib import Path

shutil.copy("/content/drive/MyDrive/best.onnx", "/content/best.onnx")
print("Copied")
print("Size:", Path("/content/best.onnx").stat().st_size / 1e6, "MB")

Copied
Size: 38.167458 MB


**Recap of Week 1 Results**

**Best Validation Metrics (from m6-04-assessment)**
- **mAP@0.5:** 0.915
- **mAP@0.5:0.95:** 0.728


**Two Observed Weaknesses**
1. **Small/distant cats** — the model struggled to detect cats that were far from the camera or occupied a small portion of the image, likely because the nano/small head misses fine-grained features at low resolution.
2. **Occluded or unusual poses** — cats that were partially hidden behind objects or in unusual poses (curled up, upside-down) were sometimes missed or had poorly localised bounding boxes.

In [7]:
!unzip /content/drive/MyDrive/data.zip -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/data/DATA_CLEAN/images/40b7af64ba68ae07.jpg  
  inflating: /content/data/DATA_CLEAN/images/40bbcfa4c12e942d.jpg  
  inflating: /content/data/DATA_CLEAN/images/40c65f92012b1252.jpg  
  inflating: /content/data/DATA_CLEAN/images/40cb564b5d90c663.jpg  
  inflating: /content/data/DATA_CLEAN/images/40d5d1ea72f63df9.jpg  
  inflating: /content/data/DATA_CLEAN/images/40e0b96f83afbaf7.jpg  
  inflating: /content/data/DATA_CLEAN/images/40fa77f79675288c.jpg  
  inflating: /content/data/DATA_CLEAN/images/410966201c9423b6.jpg  
  inflating: /content/data/DATA_CLEAN/images/41145cfbae7e805e.jpg  
  inflating: /content/data/DATA_CLEAN/images/4138fc2f905d3ab8.jpg  
  inflating: /content/data/DATA_CLEAN/images/4141e50bad0cdcf3.jpg  
  inflating: /content/data/DATA_CLEAN/images/415d209bbae49f0d.jpg  
  inflating: /content/data/DATA_CLEAN/images/4161ca976a6f4a9e.jpg  
  inflating: /content/data/DATA_CLEAN/images/41895b2ea6a63ded.jpg

In [8]:
import onnxruntime as ort
import numpy as np
from PIL import Image
from pathlib import Path

session = ort.InferenceSession("/content/best.onnx",
            providers=["CPUExecutionProvider"])

input_name = session.get_inputs()[0].name
output_shape = session.get_outputs()[0].shape

print(f"Input name  : {input_name}")
print(f"Output shape: {output_shape}")

IMG_DIR = Path("/content/data/DATA_CLEAN/images")
test_images = list(IMG_DIR.iterdir())[:3]

for img_path in test_images:
    img = Image.open(img_path).convert("RGB").resize((640, 640))
    img_array = np.array(img).astype(np.float32) / 255.0
    img_array = img_array.transpose(2, 0, 1)
    img_array = np.expand_dims(img_array, 0)

    outputs = session.run(None, {input_name: img_array})
    preds = outputs[0][0]

    conf_thresh = 0.25
    valid = preds[preds[:, 4] >= conf_thresh]

    print(f"{img_path.name}: {len(valid)} detections")
    for det in valid[:3]:
        x1, y1, x2, y2, conf, cls = det
        print(f"  box=[{x1:.1f},{y1:.1f},{x2:.1f},{y2:.1f}] conf={conf:.3f} class={int(cls)}")

Input name  : images
Output shape: [1, 300, 6]
20786150965848d0.jpg: 1 detections
  box=[87.6,145.6,604.5,513.8] conf=0.959 class=0
02aa416752a4efd1.jpg: 1 detections
  box=[110.0,229.3,569.8,510.6] conf=0.583 class=0
33bcffdbf9cd1917.jpg: 1 detections
  box=[-3.3,78.4,593.5,648.8] conf=0.909 class=0


In [9]:
import os

# Create directories
os.makedirs("/content/container/app", exist_ok=True)
os.makedirs("/content/container/models", exist_ok=True)

# Create empty __init__.py
open("/content/container/app/__init__.py", "w").close()

# Copy ONNX model
import shutil
shutil.copy("/content/best.onnx", "/content/container/models/best.onnx")

print("Directory structure created:")
for root, dirs, files in os.walk("/content/container"):
    level = root.replace("/content/container", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

Directory structure created:
container/
  models/
    best.onnx
  app/
    __init__.py


In [10]:
import json

student = {
    "first_name": "Khavar",
    "last_name": "Gasimova",
    "team": "khavar-gasimova",
    "model": {
        "framework": "yolo26",
        "variant": "yolo26s",
        "imgsz": 640,
        "epochs_total": 30,
        "tricks": []
    },
    "notes": "Week-1 yolo26s baseline, mAP@0.5=0.915"
}

with open("/content/container/STUDENT.json", "w") as f:
    json.dump(student, f, indent=2)

print("STUDENT.json written:")
print(json.dumps(student, indent=2))

STUDENT.json written:
{
  "first_name": "Khavar",
  "last_name": "Gasimova",
  "team": "khavar-gasimova",
  "model": {
    "framework": "yolo26",
    "variant": "yolo26s",
    "imgsz": 640,
    "epochs_total": 30,
    "tricks": []
  },
  "notes": "Week-1 yolo26s baseline, mAP@0.5=0.915"
}


In [20]:
detector_code = '''import numpy as np
import onnxruntime as ort
from PIL import Image


class CatDetector:
    def __init__(self, onnx_path="/app/models/best.onnx", imgsz=640, conf=0.25, class_names=("cat",)):
        self.session = ort.InferenceSession(
            onnx_path,
            providers=["CPUExecutionProvider"]
        )
        self.imgsz = imgsz
        self.conf = conf
        self.class_names = class_names
        self.input_name = self.session.get_inputs()[0].name

    def _letterbox(self, img, imgsz):
        """Resize image with padding to preserve aspect ratio."""
        orig_w, orig_h = img.size
        scale = min(imgsz / orig_w, imgsz / orig_h)
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        img_resized = img.resize((new_w, new_h), Image.BILINEAR)

        # Create padded image
        padded = Image.new("RGB", (imgsz, imgsz), (114, 114, 114))
        pad_x = (imgsz - new_w) // 2
        pad_y = (imgsz - new_h) // 2
        padded.paste(img_resized, (pad_x, pad_y))

        return padded, scale, (pad_x, pad_y)

    def predict(self, image_path: str) -> list:
        img = Image.open(image_path).convert("RGB")
        orig_w, orig_h = img.size

        # Letterbox preprocessing
        x, scale, (pad_x, pad_y) = self._letterbox(img, self.imgsz)
        x = (np.array(x, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None, ...]

        # Run inference
        out = self.session.run(None, {self.input_name: x})[0]  # (1, 300, 6)
        out = out[0]  # (300, 6)

        results = []
        for x1, y1, x2, y2, score, cls in out:
            if float(score) < self.conf:
                continue
            # Undo letterbox -> original image pixels
            x1 = (float(x1) - pad_x) / scale
            y1 = (float(y1) - pad_y) / scale
            x2 = (float(x2) - pad_x) / scale
            y2 = (float(y2) - pad_y) / scale
            # Clip to image bounds
            x1 = max(0.0, min(orig_w, x1))
            y1 = max(0.0, min(orig_h, y1))
            x2 = max(0.0, min(orig_w, x2))
            y2 = max(0.0, min(orig_h, y2))
            results.append({
                "xmin": x1,
                "ymin": y1,
                "xmax": x2,
                "ymax": y2,
                "confidence": float(score),
                "class": self.class_names[int(cls)],
            })
        return results
'''

with open("/content/container/app/detector.py", "w") as f:
    f.write(detector_code)

print("detector.py updated with letterbox")

detector.py updated with letterbox


In [24]:
cli_code = '''import argparse
import csv
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, "/app")
from app.detector import CatDetector


def cmd_info():
    with open("/app/STUDENT.json", "r") as f:
        print(f.read())


def cmd_predict():
    input_dir  = Path("/data/input")
    output_dir = Path("/data/output")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_csv = output_dir / "predictions.csv"

    detector = CatDetector(
        onnx_path="/app/models/best.onnx",
        imgsz=640,
        conf=0.25
    )

    img_exts = {".jpg", ".jpeg", ".png"}
    image_paths = sorted([
        p for p in input_dir.rglob("*")
        if p.suffix.lower() in img_exts
    ])

    print(f"Found {len(image_paths)} images", flush=True)

    with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["image_path", "xmin", "ymin", "xmax", "ymax", "confidence", "class"])

        for img_path in image_paths:
            rel_path = img_path.relative_to(input_dir).as_posix()
            detections = detector.predict(str(img_path))

            if not detections:
                writer.writerow([rel_path, "", "", "", "", "", ""])
            else:
                for det in detections:
                    writer.writerow([
                        rel_path,
                        det["xmin"],
                        det["ymin"],
                        det["xmax"],
                        det["ymax"],
                        det["confidence"],
                        det["class"]
                    ])

    print(f"Predictions saved to {output_csv}", flush=True)


def main():
    parser = argparse.ArgumentParser(description="Cat detector CLI")
    parser.add_argument("command", choices=["info", "predict"])
    args = parser.parse_args()

    if args.command == "info":
        cmd_info()
    elif args.command == "predict":
        cmd_predict()


if __name__ == "__main__":
    main()
'''

with open("/content/container/app/cli.py", "w") as f:
    f.write(cli_code)

print("cli.py fixed")

cli.py fixed


In [25]:
requirements = '''onnxruntime==1.21.0
numpy==1.26.4
pillow==10.4.0
opencv-python-headless==4.10.0.84
'''

with open("/content/container/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt updated")

requirements.txt updated


In [26]:
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY app /app/app
COPY models /app/models
COPY STUDENT.json /app/STUDENT.json

ENTRYPOINT ["python", "/app/app/cli.py"]
'''

with open("/content/container/Dockerfile", "w") as f:
    f.write(dockerfile)

print("Dockerfile updated")

Dockerfile updated


In [27]:
import os

print("Final container structure:")
for root, dirs, files in os.walk("/content/container"):
    level = root.replace("/content/container", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

Final container structure:
container/
  Dockerfile
  requirements.txt
  STUDENT.json
  models/
    best.onnx
  app/
    cli.py
    __init__.py
    detector.py
    __pycache__/
      __init__.cpython-312.pyc
      detector.cpython-312.pyc


In [28]:
import sys
sys.path.insert(0, "/content/container")

from app.detector import CatDetector
from pathlib import Path

detector = CatDetector(
    onnx_path="/content/best.onnx",
    imgsz=640,
    conf=0.25
)

# Test on 3 images
IMG_DIR = Path("/content/data/DATA_CLEAN/images")
test_images = list(IMG_DIR.iterdir())[:3]

for img_path in test_images:
    detections = detector.predict(str(img_path))
    print(f"{img_path.name}: {len(detections)} detections")
    for det in detections:
        print(f"  box=[{det['xmin']:.1f},{det['ymin']:.1f},{det['xmax']:.1f},{det['ymax']:.1f}] conf={det['confidence']:.3f} class={det['class']}")

20786150965848d0.jpg: 1 detections
  box=[87.6,145.6,604.5,513.8] conf=0.959 class=cat
02aa416752a4efd1.jpg: 1 detections
  box=[386.6,615.2,2040.9,1395.1] conf=0.931 class=cat
33bcffdbf9cd1917.jpg: 1 detections
  box=[0.0,253.2,2404.1,1927.0] conf=0.924 class=cat


In [29]:
import shutil

shutil.make_archive(
    "/content/drive/MyDrive/container",
    "zip",
    "/content",
    "container"
)

print("container.zip saved to Google Drive")

container.zip saved to Google Drive


In [30]:
readme = """# M6-09 Assessment — Cat Detection v2

## Image for leaderboard
docker pull gasimova51/cat-detector:final
Image: gasimova51/cat-detector:final
Student: Khavar Gasimova

## Week-1 Baseline Results
- mAP@0.5: 0.915
- mAP@0.5:0.95: 0.728
- Variant: yolo26s
- Epochs: 30

## Run inference
```bash
docker run --rm gasimova51/cat-detector:final info

docker run --rm \\
  -v /path/to/images:/data/input:ro \\
  -v /path/to/output:/data/output \\
  gasimova51/cat-detector:final predict
```
"""

with open("/content/README.md", "w") as f:
    f.write(readme)

print("README.md written")

README.md written


In [31]:
import shutil

# Copy README to Drive
shutil.copy("/content/README.md", "/content/drive/MyDrive/README.md")

# Re-zip container folder with updated files
shutil.make_archive(
    "/content/drive/MyDrive/container",
    "zip",
    "/content",
    "container"
)

print("README.md and container.zip saved to Drive")

README.md and container.zip saved to Drive
